# Regime Engine v2 - Calibration Q8

**Round 8** makes the macro layer more responsive to release timing without replacing the concept basket weights.

Q7 identified that macro data and market data do not end on the same date and that monthly macro observations can remain forward-filled at full weight between releases. Q8 keeps Q6 engine weights and thresholds unchanged, then changes only macro-side freshness handling and one macro concept membership.


## Changes Applied

* Macro panel construction now writes `_age_bdays:{series_id}` columns, measuring business days since each series' last realistic release date.
* Macro aggregation applies `effective_member_weight = concept_member_weight * recency_weight` inside each concept.
* Across concepts, aggregation applies `effective_concept_weight = concept_weight * concept_availability`, so semantic concept weights remain the max weight and freshness only scales availability.
* `fred_series.yml` enables mild recency weighting: half-life `42` business days, floor `0.65`.
* `ICSA` is activated inside the `labor` concept: `UNRATE 0.35`, `PAYEMS 0.35`, `ICSA 0.30`. This adds a weekly labor-market input without creating a separate concept that would double-count labor.


## Candidate Readout

| candidate | recency config | labor basket | 2009 target | 2020H2 target | 2020H2 first target | 2022 target | 2023-24 target | note |
|---|---|---|---:|---:|---|---:|---:|---|
| no recency | disabled | `UNRATE/PAYEMS/ICSA = .35/.35/.30` | `0.265` | `0.588` | `2020-07-03` | `0.369` | `0.561` | ICSA helps, but stale monthly series still keep full weight |
| too aggressive | `21 bdays`, floor `0.25` | same | `0.265` | `0.496` | `2020-07-22` | `0.315` | `0.511` | rejected: weakens the 2020 reopening response |
| selected | `42 bdays`, floor `0.65` | same | `0.265` | `0.567` | `2020-07-06` | `0.346` | `0.502` | preserves Q6 recovery readout while adding release-date awareness |


## Anchor Results After Q8

| anchor | majority | target match | first target | lag bdays | stress | disagreement | macro G/I |
|---|---|---:|---|---:|---:|---:|---:|
| 2009-10 Recovery | Neutral/Mixed Growth / Neutral/Mixed Inflation | `0.265` | `2010-04-20` | `209` | `0.000` | `0.249` | `-0.123 / -0.313` |
| 2017 Soft Landing | Goldilocks / Expansion | `0.527` | `2017-01-02` | `0` | `0.000` | `0.285` | `0.175 / -0.399` |
| 2020 H2-2021 Reopening | Reflation | `0.567` | `2020-07-06` | `3` | `0.000` | `0.359` | `0.128 / -0.148` |
| 2022 Inflation Shock / Tightening | Neutral/Mixed Growth / Neutral/Mixed Inflation | `0.346` | `2022-02-22` | `36` | `0.000` | `0.562` | `0.304 / 0.540` |
| 2023-24 Disinflation / Soft Landing | Neutral/Mixed Growth / Neutral/Mixed Inflation | `0.502` | `2023-01-09` | `5` | `0.000` | `0.375` | `-0.022 / 0.502` |
| 2025 April Liberation Day Tariff Shock | Down Growth / Neutral/Mixed Inflation | `0.705` | `2025-04-03` | `2` | `0.114` | `0.000` | `0.050 / 0.166` |


## Decision

Select mild recency weighting plus weekly claims inside the labor concept. The stronger 21-bday decay looked cleaner conceptually but was empirically too aggressive: it delayed the 2020H2 reopening target and reduced target share. Q8 should not be treated as risk-overlay tuning; GFC / 2018 Q4 / April 2025 risk remains a separate pass.


## Validation

* `conda run -n py313 python -m pytest tests/unit/regimes/test_macro_regime.py tests/unit/data_sources/test_fred_macro_panel.py -q`
* `conda run -n py313 python -m market_helper.cli.main regime-calibrate`
* `conda run -n py313 python -m market_helper.cli.main regime-run-report --methods all --output-regime data/artifacts/regime_detection/regime_snapshots.json --output-html data/artifacts/regime_detection/regime_report.html`
